In [1]:
import numpy as np

# --- PARÁMETROS FIJOS (Física de Materiales) ---
J_max = 40.0         # A/cm^2 (Límite de emisión del Tungsteno)
lambda_ion = 3.0     # cm 
alpha_loss = 0.01    # Pérdida térmica
r_lim_mag = 5.5      # cm 
limite_esbeltez = 10.0 # L / rc <= 10

# --- RESTRICCIÓN DE PRESUPUESTO DE LABORATORIO ---
# Penalizamos exponencialmente los diseños que superen los 500 A 
# (500 A es el límite típico de fuentes de poder comerciales/universitarias)
I_presupuesto = 500.0 

# --- ESPACIO 4D (Reducimos puntos a 30 para que no explote la RAM) ---
I_vals = np.linspace(100.0, 1200.0, 30) # Corriente de 100 a 1200 A
rc_vals = np.linspace(0.1, 4.0, 30)
ra_vals = np.linspace(1.0, 10.0, 30)
L_vals = np.linspace(1.0, 20.0, 30)

max_eficiencia = -1.0
mejor_I, mejor_rc, mejor_ra, mejor_L = 0, 0, 0, 0

print("Calculando matriz 4D (810,000 combinaciones)... esto tomará unos segundos.")

for I_arc in I_vals:
    for rc in rc_vals:
        for ra in ra_vals:
            if ra <= rc: continue 
                
            for L in L_vals:
                # 1. RESTRICCIÓN ESTRUCTURAL (Evitar pandeo)
                if (L / rc) > limite_esbeltez:
                    continue 
                    
                # 2. EVALUACIÓN TÉRMICA (Evitar que se derrita a esta corriente)
                area_emision = 2 * np.pi * rc * (0.5 * L) 
                J_actual = I_arc / area_emision
                if J_actual > J_max:
                    continue 
                    
                # 3. EVALUACIÓN FÍSICA (Empuje, Confinamiento, Fluidos)
                empuje_log = np.log(ra / rc)
                confinamiento = np.exp(-ra / r_lim_mag)
                ionizacion = (1 - np.exp(-L / lambda_ion))
                friccion = (1 - alpha_loss * L)
                
                rendimiento_fisico = empuje_log * confinamiento * ionizacion * friccion
                
                # 4. LA FUNCIÓN DE COSTO (El "Juez" de Presupuesto)
                # Penaliza fuertemente si pide mucha más corriente que nuestro presupuesto
                costo_laboratorio = np.exp(-I_arc / I_presupuesto)
                
                # MÉRITO GLOBAL (Física pura vs Viabilidad económica)
                eficiencia_total = rendimiento_fisico * costo_laboratorio
                
                if eficiencia_total > max_eficiencia:
                    max_eficiencia = eficiencia_total
                    mejor_I, mejor_rc, mejor_ra, mejor_L = I_arc, rc, ra, L

# --- RESULTADOS PARA TU PAPER ---
flujo_masico_mgs = (mejor_I / 1000.0)**2 / 20.0 * 1000.0 # m_dot = (I_kA^2 / 20) en mg/s

print("\n==================================================")
print("   DISEÑO ÓPTIMO 4D (FÍSICA + VIABILIDAD LAB)")
print("==================================================")
print(f"-> Corriente de Arco (I) : {mejor_I:.0f} Amperios")
print(f"-> Radio del Cátodo (rc) : {mejor_rc:.3f} cm")
print(f"-> Radio del Ánodo  (ra) : {mejor_ra:.3f} cm")
print(f"-> Longitud Total   (L)  : {mejor_L:.3f} cm")
print("--------------------------------------------------")
print(f"-> Flujo Másico (Argón)  : {flujo_masico_mgs:.2f} mg/s (Por Ley de Onset)")
print("==================================================")

Calculando matriz 4D (810,000 combinaciones)... esto tomará unos segundos.

   DISEÑO ÓPTIMO 4D (FÍSICA + VIABILIDAD LAB)
-> Corriente de Arco (I) : 100 Amperios
-> Radio del Cátodo (rc) : 0.369 cm
-> Radio del Ánodo  (ra) : 2.862 cm
-> Longitud Total   (L)  : 3.621 cm
--------------------------------------------------
-> Flujo Másico (Argón)  : 0.50 mg/s (Por Ley de Onset)


In [1]:
import numpy as np

# --- PARÁMETROS FIJOS ---
J_max = 40.0         # A/cm^2 (Límite del Tungsteno)
lambda_ion = 3.0     # cm 
alpha_loss = 0.01    # Pérdida térmica
r_lim_mag = 5.5      # cm 
limite_esbeltez = 10.0 

# --- RESTRICCIÓN DURA DE LABORATORIO ---
I_lab = 222.0 # Amperios (Compramos una máquina de 500A, no podemos pasar de aquí)

# --- ESPACIO 3D (Buscando la geometría óptima para 500A) ---
rc_vals = np.linspace(0.1, 4.0, 100)
ra_vals = np.linspace(1.0, 10.0, 100)
L_vals = np.linspace(1.0, 20.0, 100)

max_eficiencia = -1.0
mejor_rc, mejor_ra, mejor_L = 0, 0, 0

for rc in rc_vals:
    for ra in ra_vals:
        if ra <= rc: continue 
            
        for L in L_vals:
            # 1. RESTRICCIÓN ESTRUCTURAL
            if (L / rc) > limite_esbeltez: continue 
                
            # 2. EVALUACIÓN TÉRMICA (Soportar los 500A del laboratorio)
            area_emision = 2 * np.pi * rc * (0.5 * L) 
            J_actual = I_lab / area_emision
            if J_actual > J_max: continue 
                
            # 3. EVALUACIÓN FÍSICA (Maximizando el empuje para la energía que tenemos)
            empuje_log = np.log(ra / rc)
            confinamiento = np.exp(-ra / r_lim_mag)
            ionizacion = (1 - np.exp(-L / lambda_ion))
            friccion = (1 - alpha_loss * L)
            
            eficiencia_total = empuje_log * confinamiento * ionizacion * friccion
            
            if eficiencia_total > max_eficiencia:
                max_eficiencia = eficiencia_total
                mejor_rc, mejor_ra, mejor_L = rc, ra, L

# --- RESULTADOS DEL MOTOR ESCALA-LABORATORIO ---
# Ley de Onset ajustada a la nueva escala: m_dot = (I_kA^2 / 20)
flujo_masico_mgs = (I_lab / 1000.0)**2 / 20.0 * 1000.0 

print("==================================================")
print("   DISEÑO ÓPTIMO ESCALA LABORATORIO (222 Amperios)")
print("==================================================")
print(f"-> Radio del Cátodo (rc) : {mejor_rc:.3f} cm")
print(f"-> Radio del Ánodo  (ra) : {mejor_ra:.3f} cm")
print(f"-> Longitud Total   (L)  : {mejor_L:.3f} cm")
print("--------------------------------------------------")
print(f"-> Flujo Másico (Argón)  : {flujo_masico_mgs:.2f} mg/s")
print("==================================================")

   DISEÑO ÓPTIMO ESCALA LABORATORIO (222 Amperios)
-> Radio del Cátodo (rc) : 0.455 cm
-> Radio del Ánodo  (ra) : 2.909 cm
-> Longitud Total   (L)  : 4.455 cm
--------------------------------------------------
-> Flujo Másico (Argón)  : 2.46 mg/s
